# FCA–MoE v0.3 — Real GPT-OSS trace validation

This notebook validates an existing Parquet trace directory on Google Drive. It does not load the language model or fabricate routing observations. It checks the checkpoint manifest, 24-layer coverage, Top-4 expert IDs, routing-weight normalization, and token-position completeness.

In [ ]:
%pip -q install -U pandas pyarrow numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
REPO_DIR = pathlib.Path('/content/FCA-MoE')
if not REPO_DIR.exists():
    !git clone --depth 1 --branch research/fca-moe-v0.3 https://github.com/msellamiTN/FCA-MoE.git /content/FCA-MoE
%cd /content/FCA-MoE/research/v03

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/FCA_MoE_v03/gpt_oss_20b_wikitext_test"
!python scripts/validate_gpt_oss_traces.py --output-dir "$OUTPUT_DIR"

In [ ]:
import json, pandas as pd
from pathlib import Path
root = Path(OUTPUT_DIR)
manifest = json.loads((root / "manifest.json").read_text())
report = json.loads((root / "validation_report.json").read_text())
print("Manifest status:", manifest["status"])
print("Completed chunks:", manifest["completed_chunks"])
print("Rows written:", manifest["rows_written"])
display(pd.DataFrame([{
    "model": report["model_id"],
    "revision": report["model_revision"],
    "sequences": report["sequences"],
    "tokens/sequence": report["tokens_per_sequence"],
    "layer-token rows": report["layer_token_rows"],
    "layers": report["layers"],
    "Top-K": report["top_k"],
    "validation": report["status"]
}]))

## Interpretation boundary

These traces are **teacher-forced prefill** observations from a real model forward pass. They are not autoregressive decode traces. The dataset stores actual expert selections per token and layer; it does not by itself establish higher-order significance, KV-cache reuse, or inference speedups. Those require separate analyses and systems measurements.